# Feature Extraction and Integration

This notebook extracts HOG, RGB Color Histogram, and Local Binary Pattern (LBP) features from the preprocessed 20-species subset of the CUB-200-2011 dataset. It then combines these features and prepares the final training, validation, and test matrices.

In [ ]:
# Check if running in Google Colab and set up environment
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    print("Running in Google Colab. Cloning repository...")
    !git clone https://github.com/01329582993/Bird-Species-Classification-ML.git
    %cd Bird-Species-Classification-ML
else:
    print("Running locally. Working directory:", os.getcwd())

In [ ]:
# Install dependencies if needed
!pip install -q scikit-image kagglehub opencv-python Pillow

In [ ]:
# Ensure dataset is preprocessed and metadata exists
import os
from pathlib import Path
import shutil
import pandas as pd
import numpy as np

metadata_path = Path("metadata_preprocessed.csv")
dataset_preprocessed_dir = Path("dataset_20_species_preprocessed")

if not metadata_path.exists() or not dataset_preprocessed_dir.exists():
    print("Preprocessed dataset or metadata not found. Running dataset preparation...")
    import kagglehub
    download_path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
    DATASET_ROOT = download_path / "CUB_200_2011"
    IMAGES_FOLDER = DATASET_ROOT / "images"
    
    SUBSET_DIR = "./dataset_20_species"
    if os.path.exists(SUBSET_DIR):
        shutil.rmtree(SUBSET_DIR)
    os.makedirs(SUBSET_DIR)
    
    species_folders = sorted([f for f in IMAGES_FOLDER.iterdir() if f.is_dir()])
    selected_species = species_folders[:20]
    for species_path in selected_species:
        shutil.copytree(str(species_path), os.path.join(SUBSET_DIR, species_path.name))
        
    classes = pd.read_csv(DATASET_ROOT / "classes.txt", sep=r"\s+", names=["class_id", "class_name"])
    images = pd.read_csv(DATASET_ROOT / "images.txt", sep=r"\s+", names=["image_id", "image_path"])
    image_labels = pd.read_csv(DATASET_ROOT / "image_class_labels.txt", sep=r"\s+", names=["image_id", "class_id"])
    
    selected_species_names = [p.name for p in selected_species]
    dataset_info = images.merge(image_labels, on="image_id").merge(classes, on="class_id")
    dataset_info = dataset_info[dataset_info["class_name"].isin(selected_species_names)].copy().reset_index(drop=True)
    dataset_info["full_image_path"] = dataset_info["image_path"].apply(lambda p: Path(SUBSET_DIR) / p)
    
    # Import local preprocessing helpers
    import sys
    sys.path.append(str(Path(".").resolve()))
    from src.preprocessing import create_stratified_splits, preprocess_and_save_image
    
    split_metadata = create_stratified_splits(dataset_info, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=42)
    
    if dataset_preprocessed_dir.exists():
        shutil.rmtree(dataset_preprocessed_dir)
    dataset_preprocessed_dir.mkdir(parents=True, exist_ok=True)
    
    preprocessed_paths = []
    for idx, row in split_metadata.iterrows():
        src_file = row["full_image_path"]
        rel_path = Path(row["image_path"])
        dst_file = dataset_preprocessed_dir / rel_path
        preprocess_and_save_image(src_file, dst_file, target_size=(224, 224))
        preprocessed_paths.append(str(dst_file))
        
    split_metadata["preprocessed_image_path"] = preprocessed_paths
    split_metadata[["image_id", "image_path", "class_id", "class_name", "split", "preprocessed_image_path"]].to_csv(metadata_path, index=False)
    print("Dataset preparation complete!")
else:
    print("Preprocessed dataset and metadata already exist.")

In [ ]:
# Load preprocessed metadata
metadata_df = pd.read_csv("metadata_preprocessed.csv")
print("Total images in subset:", len(metadata_df))
print(metadata_df["split"].value_counts())

In [ ]:
# Import feature extraction functions
import sys
sys.path.append(str(Path(".").resolve()))

from src.feature_extraction import (
    extract_hog_features,
    extract_color_histogram,
    extract_lbp_histogram,
    combine_features
)

In [ ]:
# Run feature extraction on all images
from tqdm.auto import tqdm
import joblib

X_hog = []
X_color = []
X_lbp = []
y = []

# Output directory for processed features
OUTPUT_DIR = Path("./processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting features from preprocessed images...")
for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df), desc="Extracting features"):
    img_path = Path(row["preprocessed_image_path"])
    
    # Extract features
    hog_feat = extract_hog_features(img_path, target_size=(128, 128))
    color_feat = extract_color_histogram(img_path, bins=32)
    lbp_feat = extract_lbp_histogram(img_path, number_of_points=8, radius=1)
    
    X_hog.append(hog_feat)
    X_color.append(color_feat)
    X_lbp.append(lbp_feat)
    y.append(row["class_id"] - 1)  # Map 1-20 class_id to 0-19 label index

# Convert to numpy arrays
X_hog = np.array(X_hog, dtype=np.float32)
X_color = np.array(X_color, dtype=np.float32)
X_lbp = np.array(X_lbp, dtype=np.float32)
y = np.array(y, dtype=np.int32)
splits = metadata_df["split"].to_numpy()

# Map folder names to label indices
unique_classes = sorted(metadata_df["class_name"].unique())
label_mapping = {class_name: idx for idx, class_name in enumerate(unique_classes)}

print("\nFeature extraction complete:")
print(f"  HOG shape   : {X_hog.shape}")
print(f"  Color shape : {X_color.shape}")
print(f"  LBP shape   : {X_lbp.shape}")
print(f"  Labels shape: {y.shape}")

In [ ]:
# Combine features (Bello's tasks)
# 1. HOG + Color Histogram
X_combined_hog_color = combine_features(X_hog, X_color)

# 2. HOG + Color Histogram + LBP
X_combined_hog_color_lbp = combine_features(X_hog, X_color, X_lbp)

print("Combined features shape:")
print(f"  HOG + Color      : {X_combined_hog_color.shape}")
print(f"  HOG + Color + LBP: {X_combined_hog_color_lbp.shape}")

In [ ]:
# Save individual and combined arrays (.npy format)
np.save(OUTPUT_DIR / "X_hog.npy", X_hog)
np.save(OUTPUT_DIR / "X_color.npy", X_color)
np.save(OUTPUT_DIR / "X_lbp.npy", X_lbp)
np.save(OUTPUT_DIR / "X_combined_hog_color.npy", X_combined_hog_color)
np.save(OUTPUT_DIR / "X_combined_hog_color_lbp.npy", X_combined_hog_color_lbp)
np.save(OUTPUT_DIR / "y_labels.npy", y)
np.save(OUTPUT_DIR / "splits.npy", splits)
joblib.dump(label_mapping, OUTPUT_DIR / "label_mapping.pkl")

print("Saved raw feature matrices and label mapping successfully!")

In [ ]:
# Prepare final split matrices (Bello's task) and save to .npz
train_mask = splits == "train"
val_mask = splits == "val"
test_mask = splits == "test"

# Function to split and save npz
def save_split_npz(filename, X):
    np.savez_compressed(
        OUTPUT_DIR / filename,
        X_train=X[train_mask],
        y_train=y[train_mask],
        X_val=X[val_mask],
        y_val=y[val_mask],
        X_test=X[test_mask],
        y_test=y[test_mask]
    )

save_split_npz("hog_features.npz", X_hog)
save_split_npz("color_features.npz", X_color)
save_split_npz("lbp_features.npz", X_lbp)
save_split_npz("combined_hog_color.npz", X_combined_hog_color)
save_split_npz("combined_hog_color_lbp.npz", X_combined_hog_color_lbp)

print("Saved train/validation/test split NPZ files successfully!")

In [ ]:
# Verify split sizes
loaded_npz = np.load(OUTPUT_DIR / "combined_hog_color_lbp.npz")
print("Verification of split shapes for Combined (HOG + Color + LBP):")
print("  X_train shape:", loaded_npz["X_train"].shape)
print("  y_train shape:", loaded_npz["y_train"].shape)
print("  X_val shape  :", loaded_npz["X_val"].shape)
print("  y_val shape  :", loaded_npz["y_val"].shape)
print("  X_test shape :", loaded_npz["X_test"].shape)
print("  y_test shape :", loaded_npz["y_test"].shape)

assert loaded_npz["X_train"].shape == (780, 8206)
assert loaded_npz["X_val"].shape == (167, 8206)
assert loaded_npz["X_test"].shape == (168, 8206)
assert len(loaded_npz["y_train"]) == 780
print("\nAll split sizes match expected shapes!")